# GenLocalNews — serviço de geração no Colab

Corre as células por ordem. No fim aparece um URL `https://....trycloudflare.com`: copia-o para `remote_url` em `config/model_selection.json` da app local.

**Antes:** em *Runtime → Change runtime type* escolhe uma GPU (A100 ou L4 dão o modelo igual ao da dissertação; T4 usa 4-bit). A tua conta do Hugging Face tem de ter aceitado as condições de https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct .

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U transformers peft accelerate bitsandbytes fastapi uvicorn huggingface_hub

## 1. Entrar no Hugging Face (token *Read* da tua conta)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 2. Criar o servidor (igual ao do Deucalion, a ler o modelo do LIAAD)

In [ ]:
%%writefile inference_server_colab.py
from __future__ import annotations
import json, sys
from contextlib import asynccontextmanager
import torch
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from huggingface_hub import hf_hub_download
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
ADAPTER = "liaad/genlocalnews-llama3.1-8b-lora"
EXPECTED_FIELDS = ("antetitulo", "titulo", "superlead", "lead", "corpo")

# O prompt é o mesmo da dissertação (prompts.py no repositório do adaptador).
sys.path.insert(0, str(__import__("pathlib").Path(hf_hub_download(ADAPTER, "prompts.py")).parent))
from prompts import SYSTEM_PROMPT, build_user_prompt  # noqa: E402

tokenizer = None
model = None
MODO = ""


def _extract_first_json_object(text):
    start = text.find("{")
    if start < 0:
        return None
    depth, in_string, escape = 0, False, False
    for i in range(start, len(text)):
        c = text[i]
        if escape:
            escape = False
            continue
        if c == "\\":
            escape = True
            continue
        if c == '"':
            in_string = not in_string
            continue
        if in_string:
            continue
        if c == "{":
            depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                return text[start:i + 1]
    return None


class GenerationRequest(BaseModel):
    segment_id: str
    text: str
    max_seq_length: int = 4096
    max_new_tokens: int = 450
    temperature: float = 0.0


@asynccontextmanager
async def lifespan(app):
    global tokenizer, model, MODO
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    tokenizer.chat_template = open(hf_hub_download(ADAPTER, "chat_template.jinja"), encoding="utf-8").read()
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.truncation_side = "left"
    tokenizer.padding_side = "left"

    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram >= 20:
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=dtype, device_map="auto")
        MODO = f"{dtype} (igual à dissertação)"
    else:
        q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_compute_dtype=torch.float16)
        model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=q, device_map="auto")
        MODO = "4-bit (GPU com pouca memória; resultados podem diferir ligeiramente)"
    model = PeftModel.from_pretrained(model, ADAPTER, is_trainable=False)
    model.eval()
    print("GenLocalNews Generation API pronta:", MODO, flush=True)
    yield


app = FastAPI(title="GenLocalNews Generation API (Colab)", version="1.0", lifespan=lifespan)


@app.get("/health")
def health():
    return {"status": "ok", "base_model": BASE_MODEL, "adapter": "llama31_8b_4096_lora",
            "modo": MODO, "device": str(model.device) if model is not None else None,
            "cuda_available": torch.cuda.is_available()}


@app.post("/generate")
def generate(request: GenerationRequest):
    if model is None:
        raise HTTPException(status_code=503, detail="Modelo ainda não carregado.")
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": build_user_prompt(request.text)}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(prompt, return_tensors="pt", truncation=True,
                    max_length=request.max_seq_length).to(model.device)
    kw = {**enc, "max_new_tokens": request.max_new_tokens, "pad_token_id": tokenizer.pad_token_id,
          "eos_token_id": tokenizer.eos_token_id, "repetition_penalty": 1.08,
          "do_sample": request.temperature > 0}
    if request.temperature > 0:
        kw.update({"temperature": request.temperature, "top_p": 0.9})
    with torch.no_grad():
        out = model.generate(**kw)
    raw = tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    cand = _extract_first_json_object(raw)
    if not cand:
        raise HTTPException(status_code=422, detail={"message": "O modelo não devolveu JSON.", "raw": raw})
    try:
        parsed = json.loads(cand)
    except json.JSONDecodeError as exc:
        raise HTTPException(status_code=422, detail={"message": f"JSON inválido: {exc}", "raw": raw})
    missing = [f for f in EXPECTED_FIELDS if f not in parsed]
    extra = [f for f in parsed if f not in EXPECTED_FIELDS]
    if missing or extra:
        raise HTTPException(status_code=422, detail={"message": "Schema inválido.", "missing": missing,
                                                     "extra": extra, "raw": parsed})
    news = {f: str(parsed.get(f, "")).strip() for f in EXPECTED_FIELDS}
    if not news["titulo"] or not news["lead"] or not news["corpo"]:
        raise HTTPException(status_code=422, detail="A saída deve conter titulo, lead e corpo.")
    return {"segment_id": request.segment_id, "model_key": "llama31_8b_4096_lora",
            "model_version": "llama31_8b_4096_lora", "result": news}


## 3. Arrancar o servidor (demora 3–6 minutos a descarregar e carregar o modelo)

In [ ]:
import subprocess, time, requests
log = open("server.log", "w")
srv = subprocess.Popen(["python", "-m", "uvicorn", "inference_server_colab:app", "--host", "0.0.0.0", "--port", "8000"],
                       stdout=log, stderr=subprocess.STDOUT)
for i in range(120):
    time.sleep(5)
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=3)
        if r.ok:
            print("Servidor pronto:", r.json()); break
    except Exception:
        pass
    if srv.poll() is not None:
        print("O servidor parou. Últimas linhas do log:"); print(open("server.log").read()[-3000:]); break
else:
    print("Ainda a carregar; corre esta célula outra vez daqui a pouco ou vê server.log")

## 4. Abrir o túnel público e obter o URL

In [ ]:
import subprocess, re, time
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
tlog = open("tunnel.log", "w")
tun = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                       stdout=tlog, stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        url = m.group(0); break
print("URL para a app:", url)

## 5. Testar

In [ ]:
import requests
seg = ("Pelo Senhor Presidente foi presente a proposta de atribuição de um apoio de 5.000 euros à associação "
       "humanitária dos bombeiros voluntários. A Câmara deliberou, por unanimidade, aprovar o apoio.")
r = requests.post(url + "/generate", json={"segment_id": "teste", "text": seg}, timeout=600)
print(r.status_code); print(r.json())

**Deixa este separador aberto** enquanto usas a app. Se a sessão do Colab terminar, corre outra vez as células 3 e 4: o URL muda e tens de o atualizar na app.